# FoundationStereo inference (Colab): zero-shot, E0, E1, E2, E3a, E3b

**What to upload to Colab:** this notebook and `foundationstereo_source_bundle.zip`. Put the zip in `/content` using the folder icon on the left, or in Google Drive under `MyDrive/`.

**Steps**
1. Runtime → Change runtime type → **GPU**.
2. Edit **cell 1 (CONFIG)**. Every place you need to change is marked with `# <<<`.
3. Runtime → **Run all**. Allow Google Drive access when asked, because the fine-tuned checkpoints are on Drive.

**Where the models come from**

| Model | Source |
|---|---|
| `zero_shot` | downloaded automatically from Hugging Face (~3.3 GB) |
| `E1`, `E2`, `E3a`, `E3b` | `MyDrive/foundationstereo_e1_e4_lora/checkpoints/` (where the E1/E2/LoRA notebook saved them) |
| `E3a`, `E3b` without the full checkpoint | rebuilt from the zero-shot model and the small LoRA adapter file |
| `E0_A`, `E0_B` | **upload these yourself** to `MyDrive/foundationstereo_models/` (`protocol_a_finetuned.pth`, `protocol_b_finetuned.pth`), or change the paths in cell 1 |

**Input modes** (`INPUT_MODE` in cell 1)

| Mode | What it does |
|---|---|
| `"middlebury_demo"` (default) | downloads Middlebury half-resolution data with ground truth. Works with no other setup. |
| `"upload"` | the notebook asks you to upload a left image, a right image and, optionally, a ground-truth disparity |
| `"pair"` | file paths to one left/right pair, e.g. on Drive |
| `"folder"` | a left folder and a right folder with matching filenames, e.g. KITTI `image_2` / `image_3` |
| `"scenes"` | a folder of scene sub-folders containing `im0.png` / `im1.png` (Middlebury layout) |

**Outputs:** the colour disparity maps are shown in the notebook. All outputs are zipped and downloaded at the end, and also copied to Drive if `COPY_TO_DRIVE = True`.

In [ ]:
# ===================================== 1. CONFIG - EDIT THIS CELL =====================================

# ---- 1a. Which models to run (one or more). Options: "zero_shot", "E0_A", "E0_B", "E1", "E2", "E3a", "E3b"
MODELS = ["zero_shot", "E1", "E3a"]                                            # <<< choose models

# ---- 1b. Where the fine-tuned checkpoints are (Google Drive paths). zero_shot is downloaded automatically.
DRIVE_CKPT_DIR = "/content/drive/MyDrive/foundationstereo_e1_e4_lora/checkpoints"   # <<< E1/E2/E3 checkpoint folder
E0_DIR = "/content/drive/MyDrive/foundationstereo_models"                            # <<< folder where you uploaded E0 files
# Drive files keep the names the training notebook gave them: E4_* = E2, E5a_* = E3a, E5b_* = E3b
MODEL_PATHS = {
    "E0_A": f"{E0_DIR}/protocol_a_finetuned.pth",         # <<< E0 Protocol A checkpoint
    "E0_B": f"{E0_DIR}/protocol_b_finetuned.pth",         # <<< E0 Protocol B checkpoint
    "E1":   f"{DRIVE_CKPT_DIR}/E1_bn_locked.pth",
    "E2":   f"{DRIVE_CKPT_DIR}/E4_gru_only.pth",
    "E3a":  f"{DRIVE_CKPT_DIR}/E5a_lora_all.pth",
    "E3b":  f"{DRIVE_CKPT_DIR}/E5b_lora_gru.pth",
}
# used only if the full E3 checkpoint above is missing (zero-shot + LoRA adapter, a few MB each)
# Drive files keep the names the training notebook gave them: E4_* = E2, E5a_* = E3a, E5b_* = E3b
LORA_ADAPTERS = {
    "E3a": f"{DRIVE_CKPT_DIR}/E5a_lora_all_lora_adapter.pth",
    "E3b": f"{DRIVE_CKPT_DIR}/E5b_lora_gru_lora_adapter.pth",
}

# ---- 1c. Input. INPUT_MODE = "middlebury_demo" | "upload" | "pair" | "folder" | "scenes"
INPUT_MODE = "middlebury_demo"                                                 # <<< choose input mode

# "middlebury_demo": which trainingH scenes ([] = all 15)
MIDDLEBURY_SCENES = ["Adirondack", "ArtL", "Jadeplant", "Pipes"]               # <<<

# "upload": you will be asked for the files when cell 6 runs
UPLOAD_GROUND_TRUTH = False          # <<< True -> also asks for a GT disparity (KITTI 16-bit png or .pfm)

# "pair": one stereo pair
LEFT_IMAGE   = "/content/drive/MyDrive/my_stereo/left.png"                     # <<< PUT LEFT IMAGE PATH HERE
RIGHT_IMAGE  = "/content/drive/MyDrive/my_stereo/right.png"                    # <<< PUT RIGHT IMAGE PATH HERE
GT_DISPARITY = None   # <<< optional, e.g. "/content/drive/MyDrive/my_stereo/disp_occ.png" (KITTI png /256) or a .pfm
GT_MASK      = None   # <<< optional mask png, pixels == 255 are evaluated (e.g. Middlebury mask0nocc.png)

# "folder": left / right folders with the same filenames (e.g. KITTI 2015 training)
LEFT_DIR  = "/content/drive/MyDrive/KITTI2015/training/image_2"                # <<< left images folder
RIGHT_DIR = "/content/drive/MyDrive/KITTI2015/training/image_3"                # <<< right images folder
GT_DIR    = None      # <<< optional, e.g. "/content/drive/MyDrive/KITTI2015/training/disp_occ_0"
PATTERN   = "*_10.png"  # <<< filename filter ("*" = all images)

# "scenes": Middlebury-style folder (<scene>/im0.png, im1.png); GT in <GT_ROOT>/<scene>/disp0GT.pfm + mask0nocc.png
SCENES_DIR = "/content/drive/MyDrive/MiddEval3/trainingH"                      # <<<
GT_ROOT    = None                                                              # <<< optional

# ---- 1d. Inference options
ITERS = 32            # GRU iterations (32 = what every evaluation in this project used)
SCALE = 1.0           # <1 downsizes the input (less memory); disparity is rescaled back to full resolution
HIERA = False         # hierarchical inference for >1K-resolution images
FP32 = False          # True disables fp16 autocast
LIMIT = 0             # only the first N pairs (0 = all)
SAVE_PFM = False      # also write .pfm disparity
SAVE_PNG16 = False    # also write KITTI-format 16-bit png (disp * 256)
FOCAL_PX = None       # set focal length (px) AND baseline (m) to also save depth maps in metres
BASELINE_M = None
SHOW_MAX_IMAGES = 6   # how many result images to display in the notebook

# ---- 1e. Output
OUT_DIR = "/content/inference_outputs"
COPY_TO_DRIVE = True
DRIVE_OUT_DIR = "/content/drive/MyDrive/foundationstereo_inference_outputs"
DOWNLOAD_ZIP = True   # download outputs.zip to your computer at the end (the .npy disparities are included)

# ======================================================================================================
VALID_MODELS = ["zero_shot", "E0_A", "E0_B", "E1", "E2", "E3a", "E3b"]
VALID_MODES = ["middlebury_demo", "upload", "pair", "folder", "scenes"]
bad = [m for m in MODELS if m not in VALID_MODELS]
assert not bad, f"Unknown model(s) {bad}; choose from {VALID_MODELS}"
assert MODELS, "MODELS is empty"
assert INPUT_MODE in VALID_MODES, f"INPUT_MODE must be one of {VALID_MODES}"
assert 0 < SCALE <= 1, "SCALE must be in (0, 1]"
print("Models    :", ", ".join(MODELS))
print("Input mode:", INPUT_MODE)
print(f"Options   : iters={ITERS} scale={SCALE} hiera={HIERA} fp32={FP32} limit={LIMIT or 'all'}")
print("Output    :", OUT_DIR, f"(+ Drive copy: {DRIVE_OUT_DIR})" if COPY_TO_DRIVE else "")

In [ ]:
# ===================================== 2. Setup (same as the working v1 notebook) =====================================
!pip -q install omegaconf open3d imageio opencv-python einops timm scipy scikit-image gdown ruamel.yaml transformations trimesh joblib pandas

import os, sys, shutil, subprocess, zipfile, glob, json
from pathlib import Path
import numpy as np
import torch

needs_drive = (COPY_TO_DRIVE or any(m != "zero_shot" for m in MODELS)
               or any(str(p).startswith("/content/drive") for p in [LEFT_IMAGE, RIGHT_IMAGE, LEFT_DIR, RIGHT_DIR, SCENES_DIR]
                      if INPUT_MODE != "middlebury_demo"))
DRIVE_OK = False
if needs_drive:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OK = True
        print("Google Drive mounted")
    except Exception as error:
        print(f"WARNING: Google Drive not mounted ({error}). Models/inputs on Drive will not be found.")

# source code: /content/foundationstereo_source_bundle.zip, or anywhere in MyDrive
candidates = [Path("/content/foundationstereo_source_bundle.zip")]
if DRIVE_OK:
    candidates += [Path(p) for p in glob.glob("/content/drive/MyDrive/**/foundationstereo_source_bundle.zip", recursive=True)][:5]
BUNDLE_ZIP = next((p for p in candidates if p.is_file()), None)
if BUNDLE_ZIP is None:
    raise FileNotFoundError("Upload foundationstereo_source_bundle.zip to /content (folder icon on the left) "
                            "or to Google Drive, then rerun this cell.")
BUNDLE = Path("/content/foundationstereo_bundle")
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
with zipfile.ZipFile(BUNDLE_ZIP) as archive:
    archive.extractall(BUNDLE)
SOURCE_ROOT = next((p.parent.parent for p in BUNDLE.rglob("core/foundation_stereo.py")), None)
if SOURCE_ROOT is None:
    raise FileNotFoundError("The source bundle does not contain core/foundation_stereo.py")
print("Source bundle:", BUNDLE_ZIP)
print("Source root  :", SOURCE_ROOT)
Path("/content/inference").mkdir(parents=True, exist_ok=True)   # for the inference script (cell 4)

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB)")
else:
    print("WARNING: no GPU - inference will run on CPU and be very slow. Runtime -> Change runtime type -> GPU.")

In [ ]:
# ===================================== 3. Zero-shot checkpoint (Hugging Face, same as v1) =====================================
# Always needed: it provides cfg.yaml, and it is the base for LoRA adapters.
from huggingface_hub import hf_hub_download

CHECKPOINT = Path("/content/checkpoints/23-51-11")
CHECKPOINT.mkdir(parents=True, exist_ok=True)
MODEL_FILE = CHECKPOINT / "model_best_bp2.pth"
CFG_FILE = CHECKPOINT / "cfg.yaml"
HF_REPO = "steve-redefine/FoundationStereoWeights"

if not CFG_FILE.exists() or CFG_FILE.stat().st_size < 100:
    print("Downloading cfg.yaml ...")
    hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename="23-51-11/cfg.yaml",
                    local_dir=CHECKPOINT.parent, local_dir_use_symlinks=False)
if not MODEL_FILE.exists() or MODEL_FILE.stat().st_size < 1_000_000_000:
    print("Downloading model_best_bp2.pth (about 3.3 GiB) ...")
    hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename="23-51-11/model_best_bp2.pth",
                    local_dir=CHECKPOINT.parent, local_dir_use_symlinks=False)
if not MODEL_FILE.exists() or MODEL_FILE.stat().st_size < 1_000_000_000:
    raise FileNotFoundError("Zero-shot checkpoint incomplete - delete it and rerun this cell.")
print(f"Zero-shot checkpoint: {MODEL_FILE} ({MODEL_FILE.stat().st_size / 1024**3:.2f} GiB)")
print("cfg.yaml            :", CFG_FILE)

In [ ]:
%%writefile /content/inference/infer.py
# ===================================== 4. Inference script (same file as inference/infer.py on your PC) =====================================
"""
FoundationStereo inference for the zero-shot model and every fine-tuned model from this project.

Pick the model(s) by name from models.json, give either one stereo pair or a folder of pairs,
and optionally ground truth to get EPE / D1 / BP-x metrics.

Examples (run from the "Computer vision" folder):

  # one pair, one model
  python inference/infer.py --model E1 --left L.png --right R.png

  # compare several models on the same pair (comma list, or "all")
  python inference/infer.py --model zero_shot,E1,E3a --left L.png --right R.png --gt disp.png

  # a KITTI 2015 folder, with metrics against disp_occ_0
  python inference/infer.py --model E1 --left_dir KITTI/training/image_2 --right_dir KITTI/training/image_3 ^
         --gt_dir KITTI/training/disp_occ_0 --pattern "*_10.png"

  # Middlebury-style folder (one sub-folder per scene with im0.png / im1.png [/ disp0GT.pfm, mask0nocc.png])
  python inference/infer.py --model all --scenes_dir MiddEval3/trainingH --gt_root MiddEval3_GT/trainingH

  python inference/infer.py --list          # show the models and whether their files are found
"""

import argparse
import csv
import fnmatch
import gc
import json
import os
import sys
import time
import types
from pathlib import Path

import cv2
import imageio.v2 as imageio
import numpy as np
import torch
import torch.nn as nn

HERE = Path(__file__).resolve().parent
DEFAULT_REPO = HERE.parent / "FoundationStereo"
DEFAULT_REGISTRY = HERE / "models.json"


# ----------------------------------------------------------------------------------------------
# Model registry
# ----------------------------------------------------------------------------------------------
def load_registry(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    models = data["models"]
    aliases = {}
    for name, entry in models.items():
        aliases[name.lower()] = name
        for alias in entry.get("aliases", []):
            aliases[alias.lower()] = name
    return data, models, aliases


def resolve_path(text, base):
    if not text:
        return None
    p = Path(os.path.expandvars(os.path.expanduser(text)))
    return p if p.is_absolute() else (base / p).resolve()


def first_existing(value, registry_dir):
    """'path'/'adapter' may be one path or a list of candidate paths; return (found or None, first candidate)."""
    candidates = [value] if isinstance(value, str) else list(value or [])
    paths = [resolve_path(c, registry_dir) for c in candidates if c]
    found = next((p for p in paths if p.is_file()), None)
    return found, (paths[0] if paths else None)


def model_files(entry, registry_dir):
    """Return (full checkpoint or None, adapter or None, expected full path, expected adapter path)."""
    full, full_expected = first_existing(entry.get("path"), registry_dir)
    adapter, adapter_expected = first_existing(entry.get("adapter"), registry_dir)
    return full, adapter, full_expected, adapter_expected


def print_model_list(models, registry_dir, base_ckpt):
    print(f"Registry: {registry_dir / 'models.json'}")
    print(f"Base (zero-shot) checkpoint: {base_ckpt}  [{'found' if base_ckpt.is_file() else 'MISSING'}]\n")
    print(f"{'name':<11} {'aliases':<28} {'status':<40} description")
    print("-" * 120)
    for name, entry in models.items():
        full, adapter, full_expected, _ = model_files(entry, registry_dir)
        if full:
            status = f"full checkpoint ({full.stat().st_size / 1024**3:.2f} GiB)"
        elif adapter and base_ckpt.is_file():
            status = "LoRA adapter + base checkpoint"
        else:
            status = f"MISSING ({full_expected.name if full_expected else '?'})"
        print(f"{name:<11} {', '.join(entry.get('aliases', [])):<28} {status:<40} {entry.get('description', '')}")


# ----------------------------------------------------------------------------------------------
# FoundationStereo import (stubs for optional packages that Utils.py imports but inference never uses)
# ----------------------------------------------------------------------------------------------
def import_foundation_stereo(repo_root):
    for optional in ["open3d", "trimesh", "joblib"]:
        try:
            __import__(optional)
        except ImportError:
            print(f"[info] '{optional}' not installed - using an empty stub (not needed for inference)")
            sys.modules[optional] = types.ModuleType(optional)
    repo_root = str(Path(repo_root).resolve())
    if not Path(repo_root, "core", "foundation_stereo.py").is_file():
        raise FileNotFoundError(f"FoundationStereo source not found at {repo_root} (use --repo_root)")
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    from omegaconf import OmegaConf
    from core.foundation_stereo import FoundationStereo
    from core.utils.utils import InputPadder
    return OmegaConf, FoundationStereo, InputPadder


# ----------------------------------------------------------------------------------------------
# LoRA (identical to the training notebook) - only used when a model is given as base + adapter
# ----------------------------------------------------------------------------------------------
class LoRAConv(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        conv = type(base)
        self.base = base
        self.lora_A = conv(base.in_channels, rank, base.kernel_size, stride=base.stride, padding=base.padding,
                           dilation=base.dilation, bias=False, padding_mode=base.padding_mode)
        self.lora_B = conv(rank, base.out_channels, 1, bias=False)
        self.scale = alpha / rank

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    @torch.no_grad()
    def merged(self):
        delta = self.lora_B.weight.flatten(1) @ self.lora_A.weight.flatten(1)
        self.base.weight += self.scale * delta.view_as(self.base.weight).to(self.base.weight.dtype)
        return self.base


class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.lora_A = nn.Linear(base.in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, base.out_features, bias=False)
        self.scale = alpha / rank

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    @torch.no_grad()
    def merged(self):
        self.base.weight += self.scale * (self.lora_B.weight @ self.lora_A.weight).to(self.base.weight.dtype)
        return self.base


def inject_lora(model, scope, rank, alpha):
    targets = []
    for parent_name, parent in model.named_modules():
        for child_name, child in parent.named_children():
            full = f"{parent_name}.{child_name}" if parent_name else child_name
            if full.startswith("feature.dino"):
                continue
            if scope is not None and full.split(".")[0] not in scope:
                continue
            if type(child) in (nn.Conv2d, nn.Conv3d) and child.groups == 1:
                targets.append((parent, child_name, LoRAConv))
            elif type(child) is nn.Linear:
                targets.append((parent, child_name, LoRALinear))
    for parent, child_name, wrapper in targets:
        setattr(parent, child_name, wrapper(getattr(parent, child_name), rank, alpha))
    return len(targets)


def apply_lora_adapter(model, adapter_path):
    blob = torch.load(adapter_path, map_location="cpu", weights_only=False)
    rank, alpha, scope, weights = blob["rank"], blob["alpha"], blob.get("scope"), blob["lora"]
    wrapped = inject_lora(model, scope, rank, alpha)
    missing, unexpected = model.load_state_dict(weights, strict=False)
    missing_lora = [k for k in missing if ".lora_A." in k or ".lora_B." in k]
    if unexpected or missing_lora:
        raise RuntimeError(f"LoRA adapter does not match the model: {len(missing_lora)} missing, "
                           f"{len(unexpected)} unexpected keys (e.g. {(missing_lora + unexpected)[:3]})")
    merged = 0
    for _, parent in list(model.named_modules()):
        for name, child in list(parent.named_children()):
            if isinstance(child, (LoRAConv, LoRALinear)):
                setattr(parent, name, child.merged())
                merged += 1
    print(f"  LoRA adapter: rank {rank}, alpha {alpha}, scope {scope or 'all'}, "
          f"{wrapped} layers wrapped, {len(weights)} tensors loaded, {merged} merged")


# ----------------------------------------------------------------------------------------------
# Model loading
# ----------------------------------------------------------------------------------------------
def build_model(fs, cfg_path, ckpt_path, adapter_path, device, fp16, iters):
    OmegaConf, FoundationStereo, _ = fs
    cfg = OmegaConf.load(cfg_path)
    cfg["vit_size"] = cfg.get("vit_size", "vitl")
    cfg["mixed_precision"] = bool(fp16)
    cfg["valid_iters"] = int(iters)
    model = FoundationStereo(cfg)

    weights_path = ckpt_path
    t0 = time.perf_counter()
    state = torch.load(weights_path, map_location="cpu", weights_only=False)
    state_dict = state["model"] if isinstance(state, dict) and "model" in state else state
    model.load_state_dict(state_dict, strict=True)
    info = {k: v for k, v in state.items() if k in ("experiment", "description", "best_step", "steps",
                                                       "global_step", "val_d1_noc", "best_val_d1_noc")} \
        if isinstance(state, dict) else {}
    del state, state_dict
    gc.collect()
    print(f"  weights: {weights_path} ({time.perf_counter() - t0:.1f}s)")
    if info:
        print("  checkpoint info: " + ", ".join(f"{k}={v}" for k, v in info.items()))
    if adapter_path:
        apply_lora_adapter(model, adapter_path)
    model.to(device).eval()
    return model


# ----------------------------------------------------------------------------------------------
# I/O
# ----------------------------------------------------------------------------------------------
def read_rgb(path):
    img = imageio.imread(path)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    img = img[..., :3]
    if img.dtype == np.uint16:
        img = (img / 257.0).round().astype(np.uint8)
    return np.ascontiguousarray(img)


def read_pfm(path):
    with open(path, "rb") as f:
        header = f.readline().strip()
        if header not in (b"Pf", b"PF"):
            raise ValueError(f"Not a PFM file: {path}")
        width, height = map(int, f.readline().split())
        scale = float(f.readline())
        values = np.fromfile(f, ("<" if scale < 0 else ">") + "f4")
    channels = 3 if header == b"PF" else 1
    data = np.flipud(values.reshape(height, width, channels) if channels == 3 else values.reshape(height, width))
    return np.ascontiguousarray(data[..., 0] if channels == 3 else data).astype(np.float32)


def write_pfm(path, disp):
    disp = np.ascontiguousarray(disp.astype(np.float32))
    with open(path, "wb") as f:
        f.write(b"Pf\n")
        f.write(f"{disp.shape[1]} {disp.shape[0]}\n".encode())
        f.write(b"-1.0\n")
        np.flipud(disp).astype("<f4").tofile(f)


def read_gt(path):
    """KITTI 16-bit PNG (value / 256, 0 = invalid), PFM (inf = invalid) or .npy. Invalid -> NaN."""
    path = Path(path)
    if path.suffix.lower() == ".pfm":
        gt = read_pfm(path)
    elif path.suffix.lower() == ".npy":
        gt = np.load(path).astype(np.float32)
    else:
        raw = imageio.imread(path)
        if raw.ndim == 3:
            raw = raw[..., 0]
        gt = raw.astype(np.float32) / (256.0 if raw.dtype == np.uint16 else 1.0)
    gt = gt.astype(np.float32)
    gt[~np.isfinite(gt) | (gt <= 0)] = np.nan
    return gt


def colorize(disp, vmin, vmax):
    valid = np.isfinite(disp)
    norm = np.zeros(disp.shape, np.uint8)
    if vmax > vmin:
        norm = ((np.nan_to_num(disp, nan=vmin) - vmin) / (vmax - vmin) * 255).clip(0, 255).astype(np.uint8)
    color = cv2.applyColorMap(norm, cv2.COLORMAP_TURBO)[:, :, ::-1].copy()
    color[~valid] = 0
    return color


def disp_range(arrays):
    values = np.concatenate([a[np.isfinite(a)].ravel() for a in arrays if a is not None and np.isfinite(a).any()]
                            or [np.zeros(1, np.float32)])
    return float(np.percentile(values, 1)), float(np.percentile(values, 99))


def label(img, text):
    img = img.copy()
    scale = max(0.5, img.shape[1] / 1600)
    thick = max(1, int(round(scale * 2)))
    (w, h), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thick)
    cv2.rectangle(img, (0, 0), (w + 12, h + 14), (0, 0, 0), -1)
    cv2.putText(img, text, (6, h + 6), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), thick, cv2.LINE_AA)
    return img


# ----------------------------------------------------------------------------------------------
# Inference + metrics
# ----------------------------------------------------------------------------------------------
def predict(model, InputPadder, left, right, device, fp16, iters, scale, hiera):
    h_full, w_full = left.shape[:2]
    if scale != 1.0:
        left = cv2.resize(left, None, fx=scale, fy=scale, interpolation=cv2.INTER_LINEAR)
        right = cv2.resize(right, None, fx=scale, fy=scale, interpolation=cv2.INTER_LINEAR)
    h, w = left.shape[:2]
    x0 = torch.from_numpy(left).to(device).float().permute(2, 0, 1)[None]
    x1 = torch.from_numpy(right).to(device).float().permute(2, 0, 1)[None]
    use_amp = fp16 and device.type == "cuda"
    model.args.mixed_precision = use_amp
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
        if hiera:
            disp = model.run_hierachical(x0, x1, iters=iters, test_mode=True, small_ratio=0.5)
        else:
            padder = InputPadder(x0.shape, divis_by=32, force_square=False)
            p0, p1 = padder.pad(x0, x1)
            disp = padder.unpad(model.forward(p0, p1, iters=iters, test_mode=True).float())
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    disp = disp.float().cpu().numpy().reshape(h, w)
    if scale != 1.0:
        disp = cv2.resize(disp, (w_full, h_full), interpolation=cv2.INTER_LINEAR) / scale
    return disp.astype(np.float32), elapsed


def compute_metrics(pred, gt, mask=None):
    valid = np.isfinite(gt) & np.isfinite(pred)
    if mask is not None:
        valid &= mask
    n = int(valid.sum())
    if n == 0:
        return None
    err = np.abs(pred[valid] - gt[valid])
    d1 = (err > 3.0) & (err / np.maximum(gt[valid], 1e-6) > 0.05)
    return {"valid_px": n, "epe": float(err.mean()), "d1": float(d1.mean() * 100),
            "bp1": float((err > 1).mean() * 100), "bp2": float((err > 2).mean() * 100),
            "bp3": float((err > 3).mean() * 100), "d1_bad_px": int(d1.sum()), "bp2_bad_px": int((err > 2).sum())}


# ----------------------------------------------------------------------------------------------
# Input pairs
# ----------------------------------------------------------------------------------------------
IMAGE_EXT = (".png", ".jpg", ".jpeg", ".ppm", ".bmp", ".tif", ".tiff")
LEFT_NAMES = ["left", "im0", "l", "left_image", "img_left", "view0"]
RIGHT_NAMES = ["right", "im1", "r", "right_image", "img_right", "view1"]
GT_NAMES = ["gt", "disp_gt", "gt_disp", "disp0gt", "disparity", "disp_occ"]
MASK_NAMES = ["mask0nocc", "mask"]


def find_by_stem(folder, stems, exts):
    files = {f.stem.lower(): f for f in sorted(folder.iterdir()) if f.is_file() and f.suffix.lower() in exts}
    return next((files[s] for s in stems if s in files), None)


def read_calibration(folder):
    """K.txt as in FoundationStereo/assets: line 1 = 3x3 K (9 numbers), line 2 = baseline in metres."""
    k_file = folder / "K.txt"
    if not k_file.is_file():
        return None
    try:
        lines = [line for line in k_file.read_text().splitlines() if line.strip()]
        K = np.array(lines[0].split(), dtype=np.float64).reshape(3, 3)
        return float(K[0, 0]), float(lines[1].split()[0])
    except Exception as error:
        print(f"[warn] could not read {k_file} ({error}) - no depth for this pair")
        return None


def collect_pairs(args):
    pairs = []  # dicts: name, left, right, gt, mask, save_dir, calib
    if args.test_images:
        root = Path(args.test_images)
        if not root.is_dir():
            sys.exit(f"ERROR: test images folder not found: {root}")
        for folder in sorted(p for p in root.iterdir() if p.is_dir()):
            left = find_by_stem(folder, LEFT_NAMES, IMAGE_EXT)
            right = find_by_stem(folder, RIGHT_NAMES, IMAGE_EXT)
            if left is None or right is None:
                print(f"[warn] {folder.name}: needs left.png + right.png (or im0.png + im1.png) - skipped")
                continue
            gt = find_by_stem(folder, GT_NAMES, IMAGE_EXT + (".pfm", ".npy"))
            mask = find_by_stem(folder, MASK_NAMES, IMAGE_EXT)
            pairs.append({"name": folder.name, "left": left, "right": right, "gt": gt, "mask": mask,
                          "save_dir": folder, "calib": read_calibration(folder)})
    elif args.left or args.right:
        if not (args.left and args.right):
            sys.exit("ERROR: --left and --right must be given together")
        pairs.append({"name": Path(args.left).stem, "left": Path(args.left), "right": Path(args.right),
                      "gt": Path(args.gt) if args.gt else None, "mask": Path(args.mask) if args.mask else None})
    elif args.left_dir or args.right_dir:
        if not (args.left_dir and args.right_dir):
            sys.exit("ERROR: --left_dir and --right_dir must be given together")
        left_dir, right_dir = Path(args.left_dir), Path(args.right_dir)
        names = sorted(f for f in os.listdir(left_dir)
                       if fnmatch.fnmatch(f, args.pattern) and f.lower().endswith((".png", ".jpg", ".jpeg", ".ppm", ".bmp")))
        for f in names:
            if not (right_dir / f).is_file():
                print(f"[warn] no right image for {f}, skipped")
                continue
            gt = Path(args.gt_dir) / f if args.gt_dir else None
            if gt is not None and not gt.is_file():
                print(f"[warn] no ground truth for {f} in {args.gt_dir} - prediction only")
                gt = None
            pairs.append({"name": Path(f).stem, "left": left_dir / f, "right": right_dir / f, "gt": gt, "mask": None})
    elif args.scenes_dir:
        root = Path(args.scenes_dir)
        for scene in sorted(p for p in root.iterdir() if p.is_dir()):
            if not ((scene / "im0.png").is_file() and (scene / "im1.png").is_file()):
                continue
            gt_dir = Path(args.gt_root) / scene.name if args.gt_root else scene
            gt = next((gt_dir / n for n in ["disp0GT.pfm", "disp0.pfm"] if (gt_dir / n).is_file()), None)
            mask = gt_dir / "mask0nocc.png" if (gt_dir / "mask0nocc.png").is_file() else None
            pairs.append({"name": scene.name, "left": scene / "im0.png", "right": scene / "im1.png", "gt": gt, "mask": mask})
    else:
        sys.exit("ERROR: give an input: --test_images, --left/--right, --left_dir/--right_dir or --scenes_dir (see --help)")
    if args.limit:
        pairs = pairs[: args.limit]
    if not pairs:
        sys.exit("ERROR: no stereo pairs found")
    for p in pairs:
        p.setdefault("save_dir", None)
        p.setdefault("calib", None)
        for key in ("left", "right"):
            if not p[key].is_file():
                sys.exit(f"ERROR: file not found: {p[key]}")
    return pairs


# ----------------------------------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------------------------------
def parse_args():
    ap = argparse.ArgumentParser(description="FoundationStereo inference with model selection",
                                 formatter_class=argparse.RawDescriptionHelpFormatter, epilog=__doc__)
    g = ap.add_argument_group("model")
    g.add_argument("--model", default="zero_shot",
                   help="name(s) from models.json, comma separated, or 'all' (default: zero_shot)")
    g.add_argument("--ckpt", help="use this checkpoint file instead of --model (any {'model': state_dict} .pth)")
    g.add_argument("--adapter", help="LoRA adapter .pth; applied on --ckpt, or on the zero-shot model if no --ckpt")
    g.add_argument("--list", action="store_true", help="list the models in the registry and exit")
    g.add_argument("--registry", default=str(DEFAULT_REGISTRY))
    g.add_argument("--repo_root", default=str(DEFAULT_REPO), help="FoundationStereo source folder")
    g.add_argument("--cfg", help="cfg.yaml (default: the one next to the zero-shot checkpoint)")

    g = ap.add_argument_group("input (choose one)")
    g.add_argument("--test_images", nargs="?", const=str(HERE / "test_images"),
                   help="folder with one sub-folder per pair (left.png + right.png, optional K.txt / gt / mask); "
                        "results are saved INTO each sub-folder. Without a value: inference/test_images")
    g.add_argument("--left"); g.add_argument("--right")
    g.add_argument("--gt", help="ground-truth disparity for the single pair (.png KITTI /256, .pfm, .npy)")
    g.add_argument("--mask", help="optional mask image; pixels == 255 are evaluated (e.g. Middlebury mask0nocc.png)")
    g.add_argument("--left_dir"); g.add_argument("--right_dir")
    g.add_argument("--gt_dir", help="ground truth folder with the same filenames as --left_dir")
    g.add_argument("--pattern", default="*", help="filename filter for --left_dir, e.g. '*_10.png' for KITTI")
    g.add_argument("--scenes_dir", help="folder of scene sub-folders containing im0.png / im1.png (Middlebury)")
    g.add_argument("--gt_root", help="for --scenes_dir: folder with <scene>/disp0GT.pfm and mask0nocc.png")
    g.add_argument("--limit", type=int, default=0, help="only the first N pairs")

    g = ap.add_argument_group("inference")
    g.add_argument("--iters", type=int, default=32, help="GRU iterations (32 = what all our evaluations used)")
    g.add_argument("--scale", type=float, default=1.0, help="downscale input by this factor (<=1) to save memory")
    g.add_argument("--hiera", action="store_true", help="hierarchical inference for >1K resolution images")
    g.add_argument("--device", default="auto", help="auto | cuda | cuda:0 | cpu")
    g.add_argument("--fp32", action="store_true", help="disable fp16 autocast (fp16 is on by default on GPU)")

    g = ap.add_argument_group("output")
    g.add_argument("--out_dir", default=None,
                   help="default inference/outputs (with --test_images: maps go into each pair folder and "
                        "metrics.csv into the test_images folder)")
    g.add_argument("--save_pfm", action="store_true", help="also write disparity as .pfm")
    g.add_argument("--save_png16", action="store_true", help="also write KITTI-format 16-bit png (disp * 256)")
    g.add_argument("--focal", type=float, help="focal length in px (with --baseline: also save depth .npy in metres)")
    g.add_argument("--baseline", type=float, help="baseline in metres")
    g.add_argument("--no_vis", action="store_true", help="skip colour images / comparison grids")
    return ap.parse_args()


def main():
    args = parse_args()
    registry_path = Path(args.registry).resolve()
    data, models, aliases = load_registry(registry_path)
    reg_dir = registry_path.parent
    base_found, base_expected = first_existing(models["zero_shot"]["path"], reg_dir)
    base_ckpt = base_found or base_expected
    cfg_path = Path(args.cfg) if args.cfg else base_ckpt.parent / "cfg.yaml"

    if args.list:
        print_model_list(models, reg_dir, base_ckpt)
        return

    # ---------- which models ----------
    runs = []  # (display name, full ckpt, adapter)
    if args.ckpt or args.adapter:
        # explicit files: --ckpt alone, --ckpt + --adapter, or --adapter alone (applied on the zero-shot base)
        ckpt = Path(args.ckpt) if args.ckpt else base_ckpt
        adapter = Path(args.adapter) if args.adapter else None
        for f in [ckpt, adapter]:
            if f is not None and not f.is_file():
                sys.exit(f"ERROR: file not found: {f}")
        runs.append(((adapter or ckpt).stem.replace("_lora_adapter", ""), ckpt, adapter))
    else:
        requested = list(models) if args.model.strip().lower() == "all" else \
            [m.strip() for m in args.model.split(",") if m.strip()]
        for req in requested:
            name = aliases.get(req.lower())
            if name is None:
                sys.exit(f"ERROR: unknown model '{req}'. Known: {', '.join(models)} (aliases: run --list)")
            full, adapter, full_expected, adapter_expected = model_files(models[name], reg_dir)
            if full:
                runs.append((name, full, None))  # full checkpoints already contain merged LoRA
            elif adapter and base_ckpt.is_file():
                print(f"[info] {name}: full checkpoint not found, using zero-shot base + LoRA adapter {adapter.name}")
                runs.append((name, base_ckpt, adapter))
            else:
                msg = f"{name}: checkpoint not found at {full_expected}"
                if adapter_expected:
                    msg += f" (nor adapter {adapter_expected})"
                if args.model.strip().lower() == "all":
                    print(f"[warn] {msg} - skipped")
                    continue
                sys.exit(f"ERROR: {msg}\n  Put the file there, or edit {registry_path}, or pass --ckpt <file>.")
    if not runs:
        sys.exit("ERROR: no model files found. Run with --list to see where they are expected.")
    if not cfg_path.is_file():
        sys.exit(f"ERROR: cfg.yaml not found at {cfg_path} (use --cfg)")

    # ---------- device ----------
    if args.device == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(args.device)
        if device.type == "cuda" and not torch.cuda.is_available():
            sys.exit("ERROR: --device cuda requested but CUDA is not available")
    fp16 = device.type == "cuda" and not args.fp32
    if not 0 < args.scale <= 1:
        sys.exit("ERROR: --scale must be in (0, 1]")

    pairs = collect_pairs(args)
    if args.out_dir:
        out_dir = Path(args.out_dir)
    elif args.test_images:
        out_dir = Path(args.test_images)
    else:
        out_dir = HERE / "outputs"

    def out_file(p, model_name, suffix):
        # --test_images: <pair folder>/<model><suffix>; otherwise <out_dir>/<model>/<pair><suffix>
        if p["save_dir"] is not None:
            return p["save_dir"] / f"{model_name}{suffix}"
        folder = out_dir / model_name
        folder.mkdir(parents=True, exist_ok=True)
        return folder / f"{p['name']}{suffix}"
    out_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 100)
    print("FoundationStereo inference")
    print(f"  models   : {', '.join(r[0] for r in runs)}")
    print(f"  pairs    : {len(pairs)} ({sum(p['gt'] is not None for p in pairs)} with ground truth)")
    gpu = f" ({torch.cuda.get_device_name(device)})" if device.type == "cuda" else ""
    print(f"  device   : {device}{gpu} | precision {'fp16 autocast' if fp16 else 'fp32'} | iters {args.iters}"
          f" | scale {args.scale} | hierarchical {args.hiera}")
    if args.test_images:
        print(f"  output   : saved inside each pair folder of {Path(args.test_images).resolve()}")
        for p in pairs:
            extra = [k for k in ("gt", "mask") if p[k]] + (["K.txt -> depth"] if p["calib"] else [])
            print(f"             {p['name']}: {p['left'].name} + {p['right'].name}"
                  + (f"  ({', '.join(extra)})" if extra else ""))
    else:
        print(f"  output   : {out_dir.resolve()}")
    print("=" * 100, flush=True)

    fs = import_foundation_stereo(args.repo_root)
    InputPadder = fs[2]

    gts = {p["name"]: (read_gt(p["gt"]) if p["gt"] else None) for p in pairs}
    masks = {p["name"]: (imageio.imread(p["mask"]) == 255 if p["mask"] else None) for p in pairs}
    metric_rows = []
    for gt_name, gt in gts.items():
        if gt is not None:
            print(f"  GT {gt_name}: {np.isfinite(gt).sum():,} valid px, range {np.nanmin(gt):.1f}-{np.nanmax(gt):.1f}"
                  + (f", mask {int(masks[gt_name].sum()):,} px" if masks[gt_name] is not None else ""))

    for run_number, (name, ckpt, adapter) in enumerate(runs, 1):
        print(f"\n[{run_number}/{len(runs)}] Loading {name} ...", flush=True)
        model = build_model(fs, cfg_path, ckpt, adapter, device, fp16, args.iters)
        for i, p in enumerate(pairs, 1):
            left, right = read_rgb(p["left"]), read_rgb(p["right"])
            if left.shape != right.shape:
                print(f"  [warn] {p['name']}: left {left.shape} and right {right.shape} differ - skipped")
                continue
            try:
                disp, sec = predict(model, InputPadder, left, right, device, fp16, args.iters, args.scale, args.hiera)
                used = "fp16" if fp16 else "fp32"
                if fp16 and not np.isfinite(disp).all():
                    print(f"  [warn] {p['name']}: non-finite output in fp16 (some GPUs, e.g. GTX 16xx) - rerunning in fp32")
                    disp, sec = predict(model, InputPadder, left, right, device, False, args.iters, args.scale, args.hiera)
                    used = "fp32"
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                sys.exit(f"ERROR: GPU out of memory on {p['name']} {left.shape[1]}x{left.shape[0]}. "
                         f"Try --scale 0.5, --hiera, or --device cpu.")

            np.save(out_file(p, name, "_disp.npy"), disp)
            if args.save_pfm:
                write_pfm(out_file(p, name, "_disp.pfm"), disp)
            if args.save_png16:
                imageio.imwrite(out_file(p, name, "_disp_kitti.png"),
                                (np.nan_to_num(disp).clip(0, 255.99) * 256).astype(np.uint16))
            calib = (args.focal, args.baseline) if (args.focal and args.baseline) else p["calib"]
            depth = None
            if calib:
                with np.errstate(divide="ignore"):
                    depth = np.where(disp > 0, calib[0] * calib[1] / disp, np.inf).astype(np.float32)
                np.save(out_file(p, name, "_depth_m.npy"), depth)
            if not args.no_vis:
                gt = gts[p["name"]]
                vmin, vmax = disp_range([gt if gt is not None else disp])
                disp_color = colorize(disp, vmin, vmax)
                imageio.imwrite(out_file(p, name, "_disp.png"), disp_color)
                # left image | disparity side by side, like the official demo's vis.png
                imageio.imwrite(out_file(p, name, "_vis.png"), np.concatenate([left, disp_color], axis=1))
                if depth is not None:
                    finite = depth[np.isfinite(depth)]
                    if finite.size:
                        near, far = np.percentile(finite, 1), np.percentile(finite, 95)
                        # near = red, far = blue (same look as the disparity map); farther than 'far' is clipped
                        imageio.imwrite(out_file(p, name, "_depth.png"),
                                        colorize(-np.clip(depth, near, far), -far, -near))

            line = f"  [{i}/{len(pairs)}] {p['name']}: {left.shape[1]}x{left.shape[0]}, {sec:.2f}s ({used}), " \
                   f"disp {np.nanmin(disp):.1f}-{np.nanmax(disp):.1f} px"
            if depth is not None and np.isfinite(depth).any():
                line += f", depth median {np.median(depth[np.isfinite(depth)]):.2f} m"
            row = {"model": name, "pair": p["name"], "seconds": round(sec, 4), "precision": used}
            gt = gts[p["name"]]
            if gt is not None:
                if gt.shape != disp.shape:
                    print(f"  [warn] {p['name']}: GT {gt.shape} vs prediction {disp.shape} - metrics skipped")
                else:
                    m = compute_metrics(disp, gt, masks[p["name"]])
                    if m:
                        row.update(m)
                        line += f" | EPE {m['epe']:.3f}  D1 {m['d1']:.2f}%  BP-2 {m['bp2']:.2f}%"
            metric_rows.append(row)
            print(line, flush=True)

        del model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    # ---------- metrics ----------
    with_gt = [r for r in metric_rows if "epe" in r]
    if metric_rows:
        keys = ["model", "pair", "seconds", "precision", "valid_px", "epe", "d1", "bp1", "bp2", "bp3",
                "d1_bad_px", "bp2_bad_px"]
        with open(out_dir / "metrics.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(metric_rows)
    if with_gt:
        print("\n" + "=" * 100)
        print("SUMMARY  (pixel-weighted EPE / D1 / BP-2 = same aggregation as the project notebooks; "
              "mean-image = average of per-image values)")
        print(f"{'model':<11} {'pairs':>5} {'EPE':>8} {'D1 %':>8} {'BP-2 %':>8} {'mean-img D1':>12} "
              f"{'mean-img BP-2':>14} {'s/pair':>7}")
        print("-" * 100)
        for name, *_ in runs:
            rows = [r for r in with_gt if r["model"] == name]
            if not rows:
                continue
            px = sum(r["valid_px"] for r in rows)
            print(f"{name:<11} {len(rows):>5} {sum(r['epe'] * r['valid_px'] for r in rows) / px:>8.4f} "
                  f"{100 * sum(r['d1_bad_px'] for r in rows) / px:>8.4f} "
                  f"{100 * sum(r['bp2_bad_px'] for r in rows) / px:>8.4f} "
                  f"{np.mean([r['d1'] for r in rows]):>12.4f} {np.mean([r['bp2'] for r in rows]):>14.4f} "
                  f"{np.mean([r['seconds'] for r in rows]):>7.2f}")
        print("Note: KITTI convention -> D1 on disp_occ (all) or disp_noc (noc); Middlebury -> BP-2 with mask0nocc.")

    # ---------- comparison grids (one per pair, all models stacked) ----------
    if not args.no_vis and len(runs) > 1:
        grid_dir = out_dir / "comparison"
        for p in pairs:
            disps = [(name, out_file(p, name, "_disp.npy")) for name, *_ in runs]
            disps = [(n, np.load(f)) for n, f in disps if f.is_file()]
            if not disps:
                continue
            gt = gts[p["name"]]
            vmin, vmax = disp_range([gt] if gt is not None else [d for _, d in disps])
            tiles = [label(read_rgb(p["left"]), f"left: {p['name']}")]
            if gt is not None and gt.shape == disps[0][1].shape:
                tiles.append(label(colorize(gt, vmin, vmax), "ground truth"))
            for n, d in disps:
                r = next((r for r in with_gt if r["model"] == n and r["pair"] == p["name"]), None)
                text = n + (f"  EPE {r['epe']:.2f}  D1 {r['d1']:.2f}%  BP-2 {r['bp2']:.2f}%" if r else "")
                tiles.append(label(colorize(d, vmin, vmax), text))
            h, w = tiles[0].shape[:2]
            tiles = [t if t.shape[:2] == (h, w) else cv2.resize(t, (w, h)) for t in tiles]
            if p["save_dir"] is not None:
                grid_path = p["save_dir"] / "comparison.png"
            else:
                grid_dir.mkdir(exist_ok=True)
                grid_path = grid_dir / f"{p['name']}_compare.png"
            imageio.imwrite(grid_path, np.concatenate(tiles, axis=0))
        print("\nComparison grids: "
              + ("comparison.png in each pair folder" if args.test_images else str(grid_dir.resolve())))

    if args.test_images:
        print(f"\nOutputs saved in each pair folder under {Path(args.test_images).resolve()}:")
        print("  <model>_disp.png (colour disparity), <model>_vis.png (left | disparity), <model>_disp.npy,")
        print("  <model>_depth.png + <model>_depth_m.npy (only if the folder has K.txt), comparison.png (if >1 model)")
        print(f"metrics.csv: {(out_dir / 'metrics.csv').resolve()}")
    else:
        print(f"Outputs: {out_dir.resolve()}  (per model: <pair>_disp.npy, _disp.png, _vis.png; metrics.csv)")


if __name__ == "__main__":
    main()

In [ ]:
# ===================================== 5. Check the selected models =====================================
REGISTRY = Path("/content/inference/models.json")
registry = {"models": {"zero_shot": {"path": str(MODEL_FILE), "aliases": ["zs"], "description": "Original FoundationStereo"}}}
for name, path in MODEL_PATHS.items():
    entry = {"path": path, "description": name}
    if name in LORA_ADAPTERS:
        entry["adapter"] = LORA_ADAPTERS[name]
    registry["models"][name] = entry
REGISTRY.write_text(json.dumps(registry, indent=2))

print(f"{'model':<10} {'status':<12} file")
print("-" * 110)
missing = []
for name in MODELS:
    if name == "zero_shot":
        print(f"{name:<10} {'OK':<12} {MODEL_FILE}")
        continue
    full = Path(MODEL_PATHS[name])
    adapter = Path(LORA_ADAPTERS[name]) if name in LORA_ADAPTERS else None
    if full.is_file():
        print(f"{name:<10} {'OK':<12} {full} ({full.stat().st_size / 1024**3:.2f} GiB)")
    elif adapter is not None and adapter.is_file():
        print(f"{name:<10} {'OK (LoRA)':<12} {adapter} -> zero-shot + adapter, merged")
    else:
        print(f"{name:<10} {'MISSING':<12} {full}" + (f"  (nor {adapter})" if adapter else ""))
        missing.append(name)
if missing:
    raise FileNotFoundError(
        f"Missing checkpoint(s) for {missing}. Fix the paths in cell 1 (MODEL_PATHS / LORA_ADAPTERS), "
        f"upload the files to Drive, or remove these models from MODELS.")
print("\nAll selected models found.")

In [ ]:
# ===================================== 6. Prepare the input =====================================
input_args = []
if INPUT_MODE == "middlebury_demo":
    import requests
    MB = Path("/content/middlebury")
    for url, name in [("https://vision.middlebury.edu/stereo/submit3/zip/MiddEval3-data-H.zip", "data-H"),
                      ("https://vision.middlebury.edu/stereo/submit3/zip/MiddEval3-GT0-H.zip", "gt0-H")]:
        archive_path, target = MB / f"{name}.zip", MB / name
        if not (target / ".extracted").exists():
            MB.mkdir(parents=True, exist_ok=True)
            print("Downloading", url)
            with requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(archive_path, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            with zipfile.ZipFile(archive_path) as z:
                z.extractall(target)
            (target / ".extracted").write_text("ok")
    data_root = next((MB / "data-H").rglob("trainingH"))
    gt_root = next((MB / "gt0-H").rglob("trainingH"))
    all_scenes = sorted(p.name for p in data_root.iterdir() if p.is_dir())
    chosen = MIDDLEBURY_SCENES or all_scenes
    unknown = [s for s in chosen if s not in all_scenes]
    assert not unknown, f"Unknown Middlebury scenes {unknown}; available: {all_scenes}"
    subset = Path("/content/middlebury_selected")
    if subset.exists():
        shutil.rmtree(subset)
    subset.mkdir()
    for scene in chosen:
        os.symlink(data_root / scene, subset / scene)
    input_args = ["--scenes_dir", str(subset), "--gt_root", str(gt_root)]
    print(f"Middlebury trainingH scenes ({len(chosen)}): {', '.join(chosen)}  (GT: disp0GT.pfm, mask0nocc)")

elif INPUT_MODE == "upload":
    from google.colab import files
    UP = Path("/content/uploads")
    UP.mkdir(exist_ok=True)
    def ask(what):
        print(f"\n>>> Choose the {what} file")
        uploaded = files.upload()
        assert len(uploaded) == 1, f"Upload exactly one {what} file"
        name, content = next(iter(uploaded.items()))
        path = UP / f"{what.split()[0]}_{name}"
        path.write_bytes(content)
        Path(name).unlink(missing_ok=True)
        print("   saved", path)
        return path
    left_path, right_path = ask("LEFT image"), ask("RIGHT image")
    input_args = ["--left", str(left_path), "--right", str(right_path)]
    if UPLOAD_GROUND_TRUTH:
        input_args += ["--gt", str(ask("GT disparity"))]

elif INPUT_MODE == "pair":
    for p in [LEFT_IMAGE, RIGHT_IMAGE] + [x for x in [GT_DISPARITY, GT_MASK] if x]:
        assert Path(p).is_file(), f"File not found: {p}  (check the path in cell 1)"
    input_args = ["--left", LEFT_IMAGE, "--right", RIGHT_IMAGE]
    if GT_DISPARITY:
        input_args += ["--gt", GT_DISPARITY]
    if GT_MASK:
        input_args += ["--mask", GT_MASK]

elif INPUT_MODE == "folder":
    for d in [LEFT_DIR, RIGHT_DIR] + ([GT_DIR] if GT_DIR else []):
        assert Path(d).is_dir(), f"Folder not found: {d}  (check the path in cell 1)"
    input_args = ["--left_dir", LEFT_DIR, "--right_dir", RIGHT_DIR, "--pattern", PATTERN]
    if GT_DIR:
        input_args += ["--gt_dir", GT_DIR]
    n = len([f for f in os.listdir(LEFT_DIR) if Path(f).match(PATTERN)])
    print(f"{n} left images match '{PATTERN}'")

elif INPUT_MODE == "scenes":
    assert Path(SCENES_DIR).is_dir(), f"Folder not found: {SCENES_DIR}"
    input_args = ["--scenes_dir", SCENES_DIR] + (["--gt_root", GT_ROOT] if GT_ROOT else [])

print("\nInput arguments:", " ".join(input_args))

In [ ]:
# ===================================== 7. Run inference =====================================
if Path(OUT_DIR).exists():
    shutil.rmtree(OUT_DIR)   # fresh output for this run
cmd = [sys.executable, "/content/inference/infer.py",
       "--model", ",".join(MODELS),
       "--registry", str(REGISTRY),
       "--repo_root", str(SOURCE_ROOT),
       "--cfg", str(CFG_FILE),
       "--iters", str(ITERS), "--scale", str(SCALE),
       "--out_dir", OUT_DIR] + input_args
if HIERA: cmd.append("--hiera")
if FP32: cmd.append("--fp32")
if LIMIT: cmd += ["--limit", str(LIMIT)]
if SAVE_PFM: cmd.append("--save_pfm")
if SAVE_PNG16: cmd.append("--save_png16")
if FOCAL_PX and BASELINE_M: cmd += ["--focal", str(FOCAL_PX), "--baseline", str(BASELINE_M)]
print("Command:\n  " + " ".join(f'"{c}"' if " " in c else c for c in cmd) + "\n", flush=True)

env = dict(os.environ, PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, cwd="/content")
for line in process.stdout:
    print(line, end="", flush=True)
if process.wait() != 0:
    raise RuntimeError(f"Inference failed (exit code {process.returncode}) - see the messages above.")
print("\nInference finished.")

In [ ]:
# ===================================== 8. Show results =====================================
import pandas as pd
import matplotlib.pyplot as plt
import imageio.v2 as imageio

out = Path(OUT_DIR)
metrics_file = out / "metrics.csv"
if metrics_file.exists():
    df = pd.read_csv(metrics_file)
    if "epe" in df and df["epe"].notna().any():
        pd.set_option("display.precision", 4)
        print("Per pair:")
        display(df[[c for c in ["model", "pair", "epe", "d1", "bp1", "bp2", "bp3", "seconds", "precision"] if c in df]])
        agg = df.dropna(subset=["epe"]).groupby("model", sort=False).apply(lambda g: pd.Series({
            "pairs": len(g),
            "EPE (px-weighted)": (g.epe * g.valid_px).sum() / g.valid_px.sum(),
            "D1 % (global)": 100 * g.d1_bad_px.sum() / g.valid_px.sum(),
            "BP-2 % (global)": 100 * g.bp2_bad_px.sum() / g.valid_px.sum(),
            "mean-image BP-2 %": g.bp2.mean(),
            "s/pair": g.seconds.mean()}))
        print("\nSummary per model (same aggregation as the project notebooks):")
        display(agg)
    else:
        display(df)

grids = sorted((out / "comparison").glob("*_compare.png"))
singles = sorted(out.glob("*/*_disp.png")) if not grids else []
to_show = (grids or singles)[:SHOW_MAX_IMAGES]
for path in to_show:
    img = imageio.imread(path)
    plt.figure(figsize=(14, 14 * img.shape[0] / img.shape[1]))
    plt.imshow(img); plt.axis("off"); plt.title(f"{path.parent.name}/{path.name}")
    plt.show()
if len(grids or singles) > SHOW_MAX_IMAGES:
    print(f"... {len(grids or singles) - SHOW_MAX_IMAGES} more images in {out}")

In [ ]:
# ===================================== 9. Save / download outputs =====================================
zip_path = shutil.make_archive("/content/inference_outputs", "zip", OUT_DIR)
print(f"Zipped outputs: {zip_path} ({Path(zip_path).stat().st_size / 1024**2:.1f} MiB)")
if COPY_TO_DRIVE and DRIVE_OK:
    import time
    dest = Path(DRIVE_OUT_DIR) / time.strftime("%Y%m%d_%H%M%S")
    shutil.copytree(OUT_DIR, dest)
    print("Copied to Drive:", dest)
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(zip_path)
print("Done.")